# 03 — Autograd and gradient descent

**Level:** Beginner · **Study time:** 45–60 minutes (including experiments)

**Prerequisites:** Notebooks 01–02; slopes at an intuitive level.

## What you will be able to do
- Compute and check a derivative.
- Explain gradient accumulation.
- Minimize a simple function with gradient descent.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 02](02_shapes_and_broadcasting.ipynb) · [Next: 04](04_linear_regression.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Differentiate a scalar expression

The derivative describes local sensitivity: how much the output changes when the input changes slightly. For `y = x² + 3x`, the derivative is `2x + 3`. Autograd records operations on tensors with `requires_grad=True`.


In [ ]:
x = torch.tensor(2., requires_grad=True)
y = x.square() + 3 * x
y.backward()
print("y:", y.item(), "dy/dx:", x.grad.item())
assert x.grad.item() == 7


## Step 3 — Check gradients numerically

A centered finite difference estimates a derivative by evaluating nearby points. It is slower and approximate, but useful for checking your reasoning. We use double precision to reduce rounding error.


In [ ]:
def f(z):
    return z.square() + 3 * z
point = torch.tensor(2., dtype=torch.float64)
eps = 1e-5
numerical = (f(point + eps) - f(point - eps)) / (2 * eps)
assert abs(numerical.item() - x.grad.item()) < 1e-5
print("Finite difference:", numerical.item())


## Step 4 — Observe accumulation

Calling backward adds to `.grad`. It does not replace the previous gradient. Recompute the forward pass to build a fresh graph; ordinary backward frees saved intermediate values.


In [ ]:
x.grad = None
f(x).backward()
first = x.grad.clone()
f(x).backward()  # A new forward graph, but the same leaf gradient buffer.
print("Once:", first.item(), "Twice:", x.grad.item())
assert torch.allclose(x.grad, 2 * first)


## Step 5 — Learn the minimum of a bowl

Gradient descent moves opposite the derivative: `parameter -= learning_rate * gradient`. We update under `no_grad` so the update itself is not recorded as part of the next learning graph.


In [ ]:
parameter = torch.tensor(-4., requires_grad=True)
history = []
for step in range(60):
    loss = (parameter - 3).square()
    loss.backward()
    with torch.no_grad():
        parameter -= 0.1 * parameter.grad
    parameter.grad = None
    history.append(loss.item())
print("Learned minimum at:", parameter.item())
assert abs(parameter.item() - 3) < 1e-3
plt.plot(history); plt.yscale("log"); plt.xlabel("Step"); plt.ylabel("Loss"); plt.show()


## Your turn — practice before opening the solutions

1. Check the gradient of `x³` at x = 2.
2. Minimize `(p + 2)²`, starting at 5.
3. Predict what `.detach()` does to `requires_grad`, then verify it. Hint: a detached tensor shares storage but leaves the graph.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
q = torch.tensor(2., requires_grad=True)
q.pow(3).backward()
assert q.grad.item() == 12
p = torch.tensor(5., requires_grad=True)
for _ in range(80):
    (p + 2).square().backward()
    with torch.no_grad():
        p -= 0.1 * p.grad
    p.grad = None
assert abs(p.item() + 2) < 1e-4
assert not q.detach().requires_grad


## Before moving on

You should be able to explain why updates need gradients, why gradients must be reset, and why evaluation should avoid building a graph.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Automatic differentiation](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)


[Course guide](../README.md) · [Previous: 02](02_shapes_and_broadcasting.ipynb) · [Next: 04](04_linear_regression.ipynb)
